In [1]:
# Install required NLP library for Bangla
!pip install indic-nlp-library transformers torch

import torch
from transformers import AutoTokenizer, AutoModel

# Test BanglaBERT Load
model_name = "csebuetnlp/banglabert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# A sample Bangla sentence
text = "আমি বাংলায় কথা বলি।"

# Tokenize and get embedding
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

# The embedding is the first token [CLS]
embedding = outputs.last_hidden_state[:, 0, :]

print(f"Embedding Shape: {embedding.shape}")
# If it prints: Embedding Shape: torch.Size([1, 768]), you are ready for Day 2!

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Shape: torch.Size([1, 768])


In [3]:
import pandas as pd

# Use the absolute path you just copied
file_path = '/nlp_dataset.csv'
df = pd.read_csv(file_path)

# 1. Check for missing values
print("--- Missing Values ---")
print(df.isnull().sum())

# 2. Check class balance
print("\n--- Class Balance ---")
print(df['Label'].value_counts())

# 3. Peek at the data
print("\n--- Data Preview ---")
print(df.head())

--- Missing Values ---
SL          0
Text        0
Type        0
Source      0
Category    0
Label       0
dtype: int64

--- Class Balance ---
Label
0    300
1    300
Name: count, dtype: int64

--- Data Preview ---
   SL                                               Text          Type  \
0   1  ঢাকা উত্তর সিটি কর্পোরেশন, প্রকৌশল বিভাগ, যান্...  Office Order   
1   2  জনাব মোঃ আরাফাত হোসেন, সহকারী প্রধান বর্জ্য ব্...  Office Order   
2   3  ঢাকা দক্ষিণ সিটি কর্পোরেশনে আগামী ১৪.০৩.২০২৫ ত...  Office Order   
3   4  প্রথমে প্রণাম করি এক করতার।\nযেই প্রভুর জীবদান...          Poem   
4   5  কত যে আঁধার পর্দা পারায়ে ভোর হল জানি না তা ।\...          Poem   

                                              Source Category  Label  
0  https://dncc.gov.bd/sites/default/files/files/...      HWT      0  
1  https://dncc.gov.bd/sites/default/files/files/...      HWT      0  
2  https://dscc.gov.bd/sites/default/files/files/...      HWT      0  
3   https://kobita.banglakosh.com/archives/4354.html    

In [11]:
# Re-defining everything cleanly
from indicnlp.tokenize import indic_tokenize

def get_handcrafted_features(text):
    tokens = indic_tokenize.trivial_tokenize(text)
    num_tokens = len(tokens)
    num_unique_tokens = len(set(tokens))
    return {
        "token_count": num_tokens,
        "unique_token_count": num_unique_tokens,
        "lexical_diversity": num_unique_tokens / num_tokens if num_tokens > 0 else 0
    }

def count_stopwords(text):
    tokens = indic_tokenize.trivial_tokenize(text)
    count = sum(1 for token in tokens if token in bangla_stopwords)
    return count

def extract_all_features(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {"token_count": 0, "unique_token_count": 0, "lexical_diversity": 0, "stopword_count": 0}
    features = get_handcrafted_features(text)
    features['stopword_count'] = count_stopwords(text)
    return features

In [12]:
# Test it on a sample to verify everything is defined
test_text = "আমি এই কাজটি করেছি।"
result = extract_all_features(test_text)
print(f"Test Result: {result}")

Test Result: {'token_count': 5, 'unique_token_count': 5, 'lexical_diversity': 1.0, 'stopword_count': 1}


In [13]:
# Apply the master function to the entire 'Text' column
features_list = df['Text'].apply(extract_all_features)

# Convert to a DataFrame
features_df = pd.json_normalize(features_list)

# Combine with the original 'Label'
# We use df[['Label']] to keep the label column
final_df = pd.concat([df[['Label']], features_df], axis=1)

# Preview your new handcrafted feature matrix
print(final_df.head())

   Label  token_count  unique_token_count  lexical_diversity  stopword_count
0      0           36                  35           0.972222               8
1      0           67                  55           0.820896               4
2      0           45                  39           0.866667               4
3      0          141                  99           0.702128               8
4      0          151                 100           0.662252              25


In [14]:
from tqdm import tqdm
import torch

# 1. Ensure model is on GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

def get_bert_embeddings_batch(texts):
    embeddings = []
    # Process in batches to be efficient
    for text in tqdm(texts):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding='max_length').to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        # Extract [CLS] token and move to CPU
        cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
        embeddings.append(cls_embedding)
    return embeddings

# 2. Extract embeddings for the whole 'Text' column
bert_embeddings = get_bert_embeddings_batch(df['Text'].tolist())

# 3. Convert to DataFrame
bert_df = pd.DataFrame(bert_embeddings)

# 4. Final Merge (Combine Handcrafted features + BERT embeddings)
# We drop 'Label' from bert_df if needed, but here we just concat.
final_full_df = pd.concat([final_df, bert_df], axis=1)

print(f"Final dataset shape: {final_full_df.shape}")

100%|██████████| 600/600 [00:21<00:00, 27.38it/s]


Final dataset shape: (600, 773)


In [16]:
# Save to CSV
#final_full_df.to_csv('final_processed_dataset.csv', index=False)

# Or save to pickle (better for keeping data types intact)
final_full_df.to_pickle('final_processed_dataset.pkl')

print("File saved successfully!")

File saved successfully!


In [5]:
# Install the Indic NLP library for your linguistic features
!pip install indic-nlp-library

# Verify installation by importing
from indicnlp.tokenize import indic_tokenize
print("Indic NLP Library ready for Day 2!")

Indic NLP Library ready for Day 2!


In [7]:
from indicnlp.tokenize import indic_tokenize

def get_handcrafted_features(text):
    # Tokenize the text
    tokens = indic_tokenize.trivial_tokenize(text)

    # Calculate some basic features
    num_tokens = len(tokens)
    num_unique_tokens = len(set(tokens))

    # Return as a dictionary (easy to convert to a row in a dataframe later)
    return {
        "token_count": num_tokens,
        "unique_token_count": num_unique_tokens,
        "lexical_diversity": num_unique_tokens / num_tokens if num_tokens > 0 else 0
    }

# Test it
test_text = "আমি বাংলায় কথা বলি।"
features = get_handcrafted_features(test_text)
print(f"Features for test text: {features}")

Features for test text: {'token_count': 5, 'unique_token_count': 5, 'lexical_diversity': 1.0}


In [8]:
# A more professional and comprehensive Bangla Stopword List
bangla_stopwords = set([
    "অতএব", "অথবা", "অধিক", "অনুযায়ী", "অনেক", "অনেকে", "অনেকেই", "অন্তত", "অন্য", "অবধি", "এখন", "এখনও",
    "আপনি", "আমরা", "ই", "এই", "এইটি", "এইসব", "উচিত", "উত্তর", "উনি", "উপর", "উপরে", "এ", "এঁরা", "এও",
    "একা", "একই", "একটি", "একটা", "এখন", "এখানে", "এখানেই", "এটা", "এটাই", "এটি", "এত", "এতটাই", "এতে",
    "এদের", "এর", "এরা", "এস", "এসে", "ঐ", "ও", "ওঁরা", "ওও", "ওকা", "ওকে", "ওদের", "ওর", "ওরা", "কখনও",
    "কত", "কবে", "কম", "কয়", "করতে", "করবেন", "করল", "করলেন", "করলো", "করা", "করাই", "করায়", "করার",
    "করি", "করিতে", "করিস", "করে", "করেই", "করেছেন", "করেছেন", "করেছে", "করেছেন", "করেন", "করোনা", "কাছ",
    "কাছে", "কাচ", "কাছেই", "কাজ", "কারও", "কারণ", "কি", "কিছু", "কিছুই", "কিন্তু", "কী", "কে", "কেউ",
    "কেউ", "কেমন", "কোনো", "কৈ", "খুব", "গেল", "গিয়েছে", "গিয়ে", "গেছে", "গেল", "গিয়ে", "চার", "চায়",
    "চাও", "চাওনা", "চলে", "ছাড়াও", "ছয়", "ছিল", "ছিলাম", "ঠিক", "তবে", "তা", "তাঁরা", "তাঁহার", "তাই",
    "তাও", "তাতে", "তাদের", "তার", "তারপর", "তারা", "তারি", "তারু", "তালা", "তি", "তিনি", "তুমি", "তো",
    "তোমার", "তৈরি", "থেকে", "থেকেই", "থাকা", "থাকেন", "থাকল", "থাকলো", "থাকেন", "দিতে", "দিন", "দিয়ে",
    "দিয়েছে", "দিয়ে", "দু", "দুই", "দুটি", "দুটো", "দেওয়া", "দেওয়ার", "দেয়", "দেখতে", "দেখা", "দিয়ে", "ধরে",
    "ধরা", "নয়", "নতুন", "নাকি", "না", "নানা", "নিজে", "নিজেই", "নিজের", "নিজেদের", "নিয়ে", "নেওয়া", "নেওয়ার",
    "নেয়", "নয়", "পর", "পর্যন্ত", "পরে", "পারি", "পেরে", "পুরো", "পেয়ে", "প্রতি", "প্রভৃতি", "প্রায়", "বলতে",
    "বললেন", "বলা", "বলাই", "বলেন", "বসে", "বহু", "বা", "বাদ", "বাইরে", "বার", "বিনা", "ভিতর", "মতো",
    "মতামত", "মধ্য", "মধ্যে", "মধ্যেই", "মাঝে", "মাত্র", "মানে", "মিলে", "যে", "যথেষ্ট", "যদি", "যদি",
    "যাও", "যা", "যাক", "যাবে", "যাই", "যাও", "যাকে", "যাতে", "যাদ", "যারা", "যিনি", "যে", "যেই",
    "যেহেতু", "সে", "সেই", "সেটা", "সেটি", "সেটাই", "সেটা", "সেও", "সেখান", "সেখানে", "সেথা", "সেথা",
    "সবার", "সবাই", "সহ", "সহজ", "সারা", "সারা", "সাথে", "সাক্ষাৎ", "সুবিধা", "সুযোগ", "সে", "সেগুলো",
    "সেখান", "সেখানে", "সেটা", "সেটি", "সেটার", "সেটাও", "সেটাই", "সৈ", "হওয়া", "হওয়ার", "হচ্ছে",
    "হয়ে", "হয়েই", "হয়েছেন", "হয়েছিল", "হয়েছে", "হয়েছেন", "হয়", "হয়নি", "হয়তো", "হয়ে", "হলো", "হয়",
    "হলে", "হলে", "হলো", "হয়নি", "হয়ে", "হচ্ছে", "হতে", "হলো"
])

def count_stopwords(text):
    tokens = indic_tokenize.trivial_tokenize(text)
    # Using set membership 'in' is O(1) time complexity, very fast!
    count = sum(1 for token in tokens if token in bangla_stopwords)
    return count

    # Test it with a sentence
test_text = "আমি এই কাজটি করেছি।"
print(f"Stopword count: {count_stopwords(test_text)}")

Stopword count: 1


In [9]:
def extract_all_features(text):
    # Check if text is valid
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {"token_count": 0, "unique_token_count": 0, "lexical_diversity": 0, "stopword_count": 0}

    # Get Handcrafted Features
    features = get_handcrafted_features(text)

    # Get Stopword Count
    features['stopword_count'] = count_stopwords(text)

    # Note: We will add embedding logic here once the dataset is ready
    return features

# Quick test
print(extract_all_features("আমি এই কাজটি করেছি।"))

{'token_count': 5, 'unique_token_count': 5, 'lexical_diversity': 1.0, 'stopword_count': 1}
